In [1]:
from ase.build import bulk
from executorlib import SlurmClusterExecutor
from jinja2 import Template
from lammpsparser import get_potential_dataframe
from pylammpsmpi import LammpsASELibrary, init_function

In [2]:
structure = bulk("Au", cubic=True).repeat((10,10,10))
element_lst = structure.get_chemical_symbols()
element_lst[0] = "Cu"
structure.set_chemical_symbols(element_lst)
structure

Atoms(symbols='Au3999Cu', pbc=True, cell=[40.8, 40.8, 40.8])

In [3]:
df_pot = get_potential_dataframe(structure)
df_pot

,Config,Filename,Model,Name,Species,Citations
3,"[pair_style eam, pair_coeff 1 1 /cmmc/ptmp/pyi...",[potential_LAMMPS/1986--Foiles-S-M--Ag-Au-Cu-N...,NISTiprpy,1986--Foiles-S-M--Ag-Au-Cu-Ni-Pd-Pt--LAMMPS--ipr1,"[Ag, Au, Cu, Ni, Pd, Pt]",[{'Foiles_1986': {'title': 'Embedded-atom-meth...
21,"[pair_style eam, pair_coeff 1 1 /cmmc/ptmp/pyi...",[potential_LAMMPS/1989--Adams-J-B--Ag-Au-Cu-Ni...,NISTiprpy,1989--Adams-J-B--Ag-Au-Cu-Ni-Pd-Pt--LAMMPS--ipr1,"[Ag, Au, Cu, Ni, Pd, Pt]",[{'Adams_1989': {'title': 'Self-diffusion and ...
31,"[pair_style eam/fs, pair_coeff * * /cmmc/ptmp/...",[potential_LAMMPS/1990--Ackland-G-J--Cu-Ag-Au-...,NISTiprpy,1990--Ackland-G-J--Cu-Ag-Au--LAMMPS--ipr1,"[Cu, Ag, Au]",[{'Ackland_1990': {'title': 'Many-body potenti...
94,"[pair_style eam/alloy, pair_coeff * * /cmmc/pt...",[potential_LAMMPS/2004--Zhou-X-W--Cu-Ag-Au--LA...,NISTiprpy,2004--Zhou-X-W--Cu-Ag-Au--LAMMPS--ipr2,"[Cu, Ag, Au]",[{'Zhou_2004': {'title': 'Misfit-energy-increa...
513,[/cmmc/ptmp/pyironhb/mambaforge/envs//pyiron_m...,[],OPENKIM,EAM_Dynamo_GolaPastewka_2018_CuAu__MO_42640331...,"[Cu, Au]",[{'Adrien_2018': {'title': 'Embedded atom meth...
594,[/cmmc/ptmp/pyironhb/mambaforge/envs//pyiron_m...,[],OPENKIM,EAM_Dynamo_ZhouJohnsonWadley_2004NISTretabulat...,"[Cu, Ag, Au]",[{'W._2004': {'title': 'Misfit-energy-increasi...
654,[/cmmc/ptmp/pyironhb/mambaforge/envs//pyiron_m...,[],OPENKIM,EMT_Asap_Standard_JacobsenStoltzeNorskov_1996_...,"[Al, Ag, Au, Cu, Ni, Pd, Pt]",[{'Jacobsen_1996': {'title': 'A semi-empirical...
905,[/cmmc/ptmp/pyironhb/mambaforge/envs//pyiron_m...,[],OPENKIM,Sim_ASAP_EMT_Rasmussen_AgAuCu__SM_847706399649...,"[Ag, Au, Cu]",[{'Jacobsen_1996': {'title': 'A semi-empirical...


In [4]:
potential = "1990--Ackland-G-J--Cu-Ag-Au--LAMMPS--ipr1"
df_pot[df_pot["Name"] == potential]["Species"].values[0], df_pot[df_pot["Name"] == potential]["Config"].values[0]

(['Cu', 'Ag', 'Au'],
 ['pair_style eam/fs',
  'pair_coeff * * /cmmc/ptmp/pyironhb/mambaforge/envs//pyiron_mpie_cmti_2026-06-17/share/iprpy/potential_LAMMPS/1990--Ackland-G-J--Cu-Ag-Au--LAMMPS--ipr1/CuAgAu.eam.fs Cu Ag Au'])

In [5]:
LAMMPS_template = """\
thermo {{thermo}}
thermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol
thermo_modify format float %20.15g
timestep {{timestep}}
velocity all create $({{velocity_rescale_factor}} * {{ temp }}) {{seed}} dist {{dist}}
fix ensemble all nvt temp {{Tstart}} {{Tstop}} {{Tdamp}}
"""

In [6]:
template = Template(LAMMPS_template)
lmp_str = template.render(
    thermo=100,
    timestep=0.001,
    velocity_rescale_factor=2.0,
    temp=300.0,
    seed=12345,
    dist="gaussian",
    Tstart=300.0,
    Tstop=300.0,
    Tdamp=0.1,
)
lmp_str

'thermo 100\nthermo_style custom step temp pe etotal pxx pxy pxz pyy pyz pzz vol\nthermo_modify format float %20.15g\ntimestep 0.001\nvelocity all create $(2.0 * 300.0) 12345 dist gaussian\nfix ensemble all nvt temp 300.0 300.0 0.1'

In [7]:
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --partition={{partition}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency %}
#SBATCH --dependency=afterok:{{ dependency | join(',') }}
{%- endif %}
{%- if memory_max %}
#SBATCH --mem={{memory_max}}G
{%- endif %}
#SBATCH --ntasks={{cores}}

{{command}}
"""

In [8]:
with SlurmClusterExecutor( 
    max_workers=1, 
    block_allocation=True,
    resource_dict={
        "submission_template": submission_template, 
        # "run_time_max": 180,  # in seconds  
        "partition": "p.cmfe",
        "cores": 80,
        "threads_per_core": 1,
    },
    init_function=init_function,
    pmi_mode="pmix",
) as exe:
    lmp = LammpsASELibrary(executor=exe, cores=80)
    lmp.interactive_structure_setter(
        structure=structure,
        units="metal",
        dimension=3,
        boundary=" ".join(["p" if coord else "f" for coord in structure.pbc]),
        atom_style="atomic",
        el_eam_lst=df_pot[df_pot["Name"] == potential]["Species"].values[0],
        calc_md=True,
    )
    for c in df_pot[df_pot["Name"] == potential]["Config"].values[0]:
        lmp.interactive_lib_command(c)
    if lmp_str is not None:
        for line in lmp_str.split("\n"):
            lmp.interactive_lib_command(line)

    lmp.interactive_lib_command("run 100")
    print(lmp.interactive_energy_pot_getter())
    lmp.close()

/cmmc/ptmp/pyironhb/mambaforge/envs/pyiron_mpie_cmti_2026-06-17/lib/python3.12/site-packages/executorlib/executor/slurm.py:192: UserWarning: The following keys are not recognized and cannot be validated: ['partition']
  validate_resource_dict_with_optional_keys(resource_dict=resource_dict)


-14956.417260034283
